# Step 4b - Add TF-IDF text to the virality model

Question 1: does the post's text add predictive power beyond the meta features (>0.55 macro-F1)?
Question 2: which specific words / phrases are associated with high vs low virality?
TF-IDF is fit *inside* cross-validation to avoid leakage.

In [1]:
# load features
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, classification_report, confusion_matrix

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(ROOT / "ml" / "data" / "video_features.parquet")

num_features = [
    "title_len_words", "desc_len_words", "trans_len_words",
    "title_sentiment", "title_has_question", "title_upper_ratio",
    "kw_price", "kw_range", "kw_charging",
    "pub_hour", "pub_dow", "pub_month",
    "duration_min", "channel_freq", "has_description", "has_transcript",
]
X = df[["text_all"] + num_features]
y = df["virality_class"]                 # 0=low, 1=medium, 2=high
print("X:", X.shape)

X: (515, 17)


In [2]:
# combined model: TF-IDF(text) + numeric meta -> RandomForest, evaluated with CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pre = ColumnTransformer([
    ("txt", TfidfVectorizer(max_features=2000, min_df=5, ngram_range=(1, 2),
                            stop_words="english", sublinear_tf=True), "text_all"),
    ("num", "passthrough", num_features),
])
model = Pipeline([
    ("pre", pre),
    ("rf", RandomForestClassifier(n_estimators=400, min_samples_leaf=3,
                                  random_state=42, n_jobs=-1)),
])

pred = cross_val_predict(model, X, y, cv=cv)
print("TF-IDF + meta  macro-F1:", round(f1_score(y, pred, average="macro"), 3))
print("(meta-only reference was ~0.55)")
print("\n", classification_report(y, pred, target_names=["low", "medium", "high"]))
print("Confusion (rows=true):\n", confusion_matrix(y, pred))

TF-IDF + meta  macro-F1: 0.547
(meta-only reference was ~0.55)

               precision    recall  f1-score   support

         low       0.62      0.65      0.64       172
      medium       0.49      0.33      0.40       171
        high       0.55      0.69      0.61       172

    accuracy                           0.56       515
   macro avg       0.55      0.56      0.55       515
weighted avg       0.55      0.56      0.55       515

Confusion (rows=true):
 [[112  30  30]
 [ 45  57  69]
 [ 23  30 119]]


In [3]:
# which words are associated with high vs low virality (interpretable, descriptive)
tfidf = TfidfVectorizer(max_features=3000, min_df=5, ngram_range=(1, 2),
                        stop_words="english", sublinear_tf=True)
Xt = tfidf.fit_transform(df["text_all"])
clf = LogisticRegression(max_iter=2000, C=1.0)
clf.fit(Xt, y)

terms = np.array(tfidf.get_feature_names_out())
for cls, name in [(2, "HIGH virality"), (0, "LOW virality")]:
    idx = list(clf.classes_).index(cls)
    top = np.argsort(clf.coef_[idx])[-15:][::-1]
    print(f"\nTop tokens -> {name}:")
    print(", ".join(terms[top]))


Top tokens -> HIGH virality:
people, http, uh, say, batteries, polestar, mean, 90, storage space, energy, gas, thousand, maybe, oil, 000

Top tokens -> LOW virality:
kia, electric, ev9, kia ev9, super bowl, km, bowl, music, mini, pov, bowl commercial, 99, ev, designed, kona
